# JijModeling 2.12.1 リリースノート

## バグ修正

### バグ修正 1：表示上は同じ型どうしが一致しないと報告される型エラーの修正

型検査時に ``Could not match actual type `T` with expected `T` ``（{external+api_reference:doc}`E-TE0004 <error_codes/error/E-TE0004>`）のように、表示上はまったく同じ二つの型が一致しないと報告される場合がある問題を修正しました。

例としては、以下のようなケースでエラーが発生していました：

In [1]:
import jijmodeling as jm


@jm.Problem.define("problem")
def problem(problem: jm.DecoratedProblem):
    T = problem.Length()
    x = problem.BinaryVar(dict_keys=jm.stream((i, t) for i in T for t in T))
    problem += problem.Constraint(
        "one-hot", jm.sum(x[i, t] for i in T for t in T) == 1
    )


instance = problem.eval({"T": 2})
instance.constraint_hints.one_hot_constraints

[OneHot(id=0, variables=[0, 1, 2, 3])]

この例は、これまで制約検出を無効にしない限り、以下のエラーで失敗していました：

```text
error[E-TE0004] Could not match actual type `T` with expected `T` on an expression `t_0_...`
```

本リリースからは問題なく評価され、上の出力のとおり one-hot 制約も検出されます。

また、制約検出が無効の場合であっても、以下のように `latex=` で独自の $\LaTeX$ 表示を指定した式について同様のエラーが発生していました：

In [2]:
problem = jm.Problem("problem")
w = problem.Float("w", ndim=1)
N = w.len_at(0)
M = w.len_at(0, latex="M")
a = problem.Float("a", shape=(N,))
b = problem.Float("b", shape=(M,))

problem.infer(a + b)

Array[w.len_at(0); float]

これまでは、`a + b` は以下のエラーになり、`jm.is_same(N, M)` は `False` を返していました：

```text
error[E-TE0015] `numeric operator +` is not supported between types `Array[w.len_at(0); float]` and `Array[w.len_at(0); float]`
```

### バグ修正 2：キー集合を内包表記で定義した場合、添え字アクセスが型エラーになることがある問題の修正

以前のリリースでは、以下の `x` のように `for` 節を二つ以上持つ内包表記で定義したストリームを `dict_keys` に指定した決定変数について、その要素を参照できるのは、キーの定義と同じ `for` 節を持つ内包表記の中に限られていました。

In [3]:
import jijmodeling as jm


@jm.Problem.define("problem")
def problem(problem: jm.DecoratedProblem):
    T = problem.Length()
    x = problem.BinaryVar(dict_keys=jm.stream((i, t) for i in T for t in T))
    problem += problem.Constraint(
        "one-hot", [jm.sum(x[i, t] for t in T) == 1 for i in T]
    )


problem

Problem(name="problem", sense=MINIMIZE, objective=0, constraints={one-hot: [Constraint(name="one-hot", lambda i: sum(stream(T).map(lambda (t: natural): x[i, t])) == 1, domain=stream(T)),],})

これまでは、上の定義は `one-hot` 制約の定義部分で以下のエラーが発生していました：

```text
error[E-TE0004] Could not match actual type `natural` with expected `T` on an expression `t_...`
```

本リリースからは、問題なく実行されるようになりました。

### バグ修正 3：シェイプが部分的に決まったプレースホルダーの $\LaTeX$ 出力の修正

`shape` の一部を `None` としてシェイプを部分的に未定にしたプレースホルダー配列についても、数理モデルの $\LaTeX$ 出力で各添え字の範囲が表示されるようになりました。
これまでは、このようなプレースホルダーは $\forall (i,j) \in \mathop{\mathtt{indices}}(c)$ のような出力に全面的にフォールバックしていました。
本リリースからは、各添え字がそれぞれの軸の範囲を動くものとして表示されます。長さが `shape` で与えられている軸にはその長さが、`None` を指定した軸には配列の `len_at` が使われます。

In [4]:
import jijmodeling as jm


@jm.Problem.define("Partially determined shape")
def problem(problem: jm.DecoratedProblem):
    N = problem.Length()
    c = problem.Float(shape=(N, None))


problem

Problem(name="Partially determined shape", sense=MINIMIZE, objective=0, constraints=[])

### バグ修正 4：`DecoratedProblem` の `element_latex` と `subscript_styles` の型ヒントの追加

`DecoratedProblem` の決定変数・プレースホルダー・`CategoryLabel`・`NamedExpr` の各コンストラクタは、実行時には `element_latex` と `subscript_styles` を受け付けていましたが、型ヒントにそれらが含まれていなかったため、正しいコードに対して型チェッカーがエラーを報告していました。
本リリースからは、これらの型ヒントが対応する `Problem` のメソッドと一致するようになりました。